# Lab 10 — SDPO On-Policy Distillation

**Prerequisites:** Lab 07 (GRPO/RLVR — group rollouts + verifiable reward) · Lecture 7 (Synthetic data & distillation, [rlhfbook.com/teach/course/lec7-chap12-synthetic-data](https://rlhfbook.com/teach/course/lec7-chap12-synthetic-data/)) · Chapter 12 incl. the OPSD section ([rlhfbook.com/c/12-synthetic-data](https://rlhfbook.com/c/12-synthetic-data)).

**The idea:** the *same* model plays both roles. The **student** samples a group of rollouts per prompt; when at least one sibling is verifiably correct, that demo is spliced into the prompt and the model is **reprompted as its own teacher**. The student's on-policy logits are distilled toward the teacher's with a hand-written **top-K reverse-KL** loss, closed with a **tail bucket** so the K+1 distribution is valid. Groups with zero correct rollouts are skipped and refilled — watch `skipped` decay as the loop converges.

This notebook is for prototyping small runs and plotting `reward` / `distill_loss` / `skipped_rate` from JSONL artifacts. The exportable, config-driven loop lives in `starter.py` (+ `train.py` once you finish the stubs).


In [ ]:
# Setup: imports + config load.
import json
from pathlib import Path

import numpy as np

import starter  # lab scaffold: signatures, constants, fixtures

CONFIG_PATH = Path("configs/10_sdpo_distillation.yaml")
cfg = starter.load_config(str(CONFIG_PATH))
print(f"model={cfg['model_name']}  K={cfg['kl_top_k']}  R={cfg['num_rollouts']}  prompts/step={cfg['prompts_per_step']}")

## Stage 1 — Group rollouts on `spell_backward`

Sample `num_rollouts` completions per prompt at the config's temperature and score each with your **string verifier** (`starter.spell_backward_reward`, hard 0/1). Reuse/rewrite your Lab 07 verifier from memory before comparing with the Lab 07 compare-pointer.

**No-solutions check:** the verifier and the group-sampling orchestration are TODO stubs in `starter.py` — implement them there first.

In [ ]:
# TODO(Lab 10): implement spell_backward_reward in starter.py, then verify it
# on the fixture groups (known structure: correct counts [1, 0, 3] of 8):
groups = starter.make_fixture_groups()
# for g in groups:
#     rewards = [starter.spell_backward_reward(g["question"], c) for c in g["completions"]]
#     assert rewards == g["rewards"], "verifier disagrees with fixture pattern"
# raise NotImplementedError

## Stage 2 — Skip-and-refill

When *zero* rollouts in a group meet `success_reward_threshold: 1.0`, discard the group, count it under the `skipped` metric, and poll a fresh prompt until `prompts_per_step` full groups are collected — with a hard polling bound (`max_polls`) so an impossible task fails fast instead of hanging.

`GroupCollector.record_group` (given plumbing) already does the per-group decision + `skipped` counter; **your TODO is `collect_full_groups`** — the poll-until-full loop.

In [ ]:
# TODO(Lab 10): implement GroupCollector.collect_full_groups in starter.py, then:
# collector = starter.GroupCollector(
#     num_rollouts=cfg["num_rollouts"],
#     prompts_per_step=cfg["prompts_per_step"],
#     success_reward_threshold=cfg["success_reward_threshold"],
#     max_polls=256,
# )
# full_groups = collector.collect_full_groups(generate_fn, sample_prompt_fn)
# print(f"collected {len(full_groups)} groups, skipped={collector.skipped}")
# raise NotImplementedError

## Stage 3 — Demonstration-conditioned teacher

Take ONE correct sibling rollout, build `question + "Correct solution:\n\n" + demo + "Correctly solve the original question."`, and rerun the *same* model on it to obtain teacher logits over exactly the student's completion tokens.

**Why a sibling demo, not a gold answer?** (Key question 2.) The teacher conditions on a token-level *trace* in the model's own style, so the teacher's continuation stays path-aligned with the student's rollout — per-token KL between two very different trajectories is noise. Decide and document how `build_teacher_prompt` selects WHICH correct sibling when several exist.

In [ ]:
# TODO(Lab 10): implement build_teacher_prompt in starter.py, then sanity-check
# the surface form on a fixture group with a correct sibling:
g = groups[0]  # exactly 1 correct rollout at index 4
demo_idx = starter.FIXTURE_CORRECT_ROLLOUTS[0][0]
assert g["rewards"][demo_idx] >= cfg["success_reward_threshold"]
# teacher_prompt = starter.build_teacher_prompt(g["question"], g["completions"][demo_idx])
# raise NotImplementedError

## Stage 4 — Top-K reverse-KL distillation loss, by hand

Project both logits onto the **student's** top-`kl_top_k` tokens (`[R, A, V] → [R, A, K]`), close each distribution with a **tail bucket** carrying all non-top-K mass (`[R, A, K+1]`, probabilities sum to 1), and compute **reverse KL = KL(student ‖ teacher)** with the gradient flowing through the student side. Chunk rollouts (`rollout_chunk`) with backward per chunk so peak memory stays bounded; chunk losses divide by the **global** action-token count so gradients accumulate to the full-group gradient.

**No-solutions check:** `TopKReverseKL.add_tail_bucket`, `_chunk_loss`, and `accumulate` are empty stubs — no KL arithmetic or top-K masking logic is shipped.

In [ ]:
# TODO(Lab 10): implement TopKReverseKL (add_tail_bucket -> _chunk_loss ->
# accumulate) in starter.py. Before coding, answer on paper:
#  1. Which KL argument order makes the gradient flow through the student?
#  2. Why gather the teacher at the STUDENT's top-K indices?
#  3. Why does dividing each chunk by the GLOBAL token count make chunk
#     gradients sum to the full-group gradient?
# loss_fn = starter.TopKReverseKL(kl_top_k=cfg["kl_top_k"], rollout_chunk=cfg["rollout_chunk"])
# raise NotImplementedError

## Stage 5 — Metrics loop

Track `reward`, `distill_loss`, and `skipped` / `skipped_rate` while the loop converges. One JSONL line per step to `cfg["metrics_path"]` (validated by `starter.validate_metrics_record`) so this notebook plots from artifacts regardless of platform.

In [ ]:
# TODO(Lab 10): implement run_training_step + run_training_loop in starter.py,
# then a small GPU prototype run (shorten num_steps first):
# metrics_path = starter.run_training_loop(model, cfg)
# raise NotImplementedError

## Plots — reward / distill_loss / skipped-rate

Run after a real (GPU) training run has produced the JSONL artifact.

In [ ]:
# TODO(Lab 10): plot from the JSONL artifact once a run exists:
# import matplotlib.pyplot as plt
# recs = [json.loads(l) for l in open(Path(cfg["metrics_path"]))]  # each validated
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, key in zip(axes, ["reward", "distill_loss", "skipped_rate"]):
#     ax.plot([r["step"] for r in recs], [r[key] for r in recs])
#     ax.set_title(key); ax.set_xlabel("step")
# plt.tight_layout(); plt.show()
# raise NotImplementedError

## Key questions (answer these here)

1. **Reverse vs forward KL** — which modes get preserved? What does the reverse-KL direction do to low-probability nonsense modes of the teacher?
   > *TODO(your answer): recall Ch. 8 — reverse KL is mode-seeking (zero-forcing): the student puts mass only where the teacher's support it overlaps has mass... spell out what happens to teacher modes the student never samples.*

2. **Why condition the teacher on a sibling demo rather than gold answers?** (Hint: token-level path/trace alignment.)
   > *TODO(your answer).*

3. **Why skip empty groups entirely?** What would distilling from an all-wrong group teach the model? Watch how `skipped` decays as training converges.
   > *TODO(your answer).*

4. **The tail bucket** means gradient never reaches teacher tokens outside the top K. When does that matter for the `spell_backward` vocabulary?
   > *TODO(your answer).*

## Compare (only after yours works)

- [`rlhf-book/code/distillation/loss.py`](../../../rlhf-book/code/distillation/loss.py) — note the `add_tail` tail-bucket trick and the chunked-forward / backward-per-chunk memory story.
- [`rlhf-book/code/distillation/configs/sdpo.yaml`](../../../rlhf-book/code/distillation/configs/sdpo.yaml)

**Report any divergence** between your hand-written loss and theirs here:

> *TODO(your divergence notes: sign/direction of KL, clamp choices in add_tail, denom semantics, chunk skip-on-nonfinite behavior, ...)*